In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv(
    "../data/processed/supervised_features.csv",
    parse_dates=["Date"]
)

df = df.sort_values(["Date", "Store"]).reset_index(drop=True)

# Same chronological split strategy as Day 9
split_date = df["Date"].quantile(0.8)

train_df = df[df["Date"] < split_date].copy()
test_df = df[df["Date"] >= split_date].copy()

feature_cols = [
    c for c in df.columns
    if c not in ["Date", "Sales"]
]

X_train = train_df[feature_cols]
y_train = train_df["Sales"]

X_test = test_df[feature_cols]
y_test = test_df["Sales"]

print("Features:", len(feature_cols))
print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Missing values:", X_train.isnull().sum().sum())

Features: 18
Train: (648146, 18)
Test: (162796, 18)
Missing values: 0


In [4]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost training complete!")

XGBoost training complete!


In [5]:
xgb_predictions = xgb_model.predict(X_test)

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.abs(y_true) + np.abs(y_pred)

    values = np.divide(
        2 * np.abs(y_true - y_pred),
        denominator,
        out=np.zeros_like(denominator),
        where=denominator != 0
    )

    return values.mean() * 100

xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = mean_squared_error(y_test, xgb_predictions) ** 0.5
xgb_smape = smape(y_test, xgb_predictions)

print("XGBoost MAE :", round(xgb_mae, 2))
print("XGBoost RMSE:", round(xgb_rmse, 2))
print("XGBoost sMAPE:", round(xgb_smape, 2), "%")

XGBoost MAE : 697.11
XGBoost RMSE: 999.25
XGBoost sMAPE: 10.32 %


In [6]:
xgb_results = pd.DataFrame([{
    "Model": "XGBoost",
    "MAE": xgb_mae,
    "RMSE": xgb_rmse,
    "sMAPE": xgb_smape
}])

xgb_results.to_csv(
    "../data/processed/day10_xgboost_evaluation.csv",
    index=False
)

predictions = test_df[["Date", "Store", "Sales"]].copy()
predictions["XGBoost_Prediction"] = xgb_predictions

predictions.to_csv(
    "../data/processed/xgboost_predictions.csv",
    index=False
)

print(xgb_results.round(2))
print("Evaluation and predictions saved!")

     Model     MAE    RMSE  sMAPE
0  XGBoost  697.11  999.25  10.32
Evaluation and predictions saved!


In [7]:
rf_results = pd.read_csv(
    "../data/processed/day9_random_forest_evaluation.csv"
)

comparison = pd.concat(
    [rf_results, xgb_results],
    ignore_index=True
)

comparison = comparison.sort_values("MAE")

comparison.round(2)

,Model,MAE,RMSE,sMAPE
1,XGBoost,697.11,999.25,10.32
0,Random Forest,710.63,1036.48,10.51


# Day 10 — XGBoost Forecasting

## Objective
Train a gradient-boosted tree model using historical sales, lag, rolling, calendar, promotion and store features.

## Model
XGBoost Regressor with histogram-based tree construction.

## Evaluation
The model was evaluated using MAE, RMSE and sMAPE on a chronological holdout.

## Outputs
- day10_xgboost_evaluation.csv
- xgboost_predictions.csv

## Model Comparison
XGBoost results were compared with the Random Forest baseline. Final model selection will depend on comparable validation results.

## Limitation
These metrics apply to the selected validation window and do not establish performance across every store or future period.